# Inferring Ligand-to-Target Signaling Paths

To determine signaling paths between a ligand and target of interest, we look at which transcription factors are best regulating the target genes and are most closely downstream of the ligand (based on the weights of the edges in the integrated ligand-signaling and gene regulatory networks). Then, the shortest paths between these transcription factors and the ligand of interest are determined and genes forming part of this path are considered as important signaling mediators. Finally, we look in our collected data source networks for all interactions between the ligand, signaling mediators, transcription factors and target genes.

In this notebook, we demonstrate how to infer signaling paths between a CAF-ligand (CAF = cancer-associated fibroblast) of interest and some of its top-predicted p-EMT target genes.

In [ ]:
import os
os.environ.setdefault("NICHENETR_DATA_DIR", "path/to/nichenetr_data")

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

import nichenetr as nn

### Load necessary networks

In [ ]:
weighted_networks = nn.load_weighted_networks("human")
ligand_tf_matrix = nn.load_ligand_tf_matrix()

lr_network = nn.load_lr_network("human")
sig_network = nn.load_sig_network()
gr_network = nn.load_gr_network()

### Infer signaling paths between ligand(s) and target(s) of interest

As an example, we infer signaling paths between the CAF-ligand TGFB2 and its top-predicted p-EMT target genes SERPINE1 and COL1A1. For better visualization of edge weights, we normalize them via min-max scaling.

In [ ]:
ligands_oi = ["TGFB2"]  # can be a list of multiple ligands
targets_oi = ["SERPINE1", "COL1A1"]

active_signaling_network = nn.get_ligand_signaling_path(
    ligand_tf_matrix=ligand_tf_matrix,
    ligands_all=ligands_oi,
    targets_all=targets_oi,
    weighted_networks=weighted_networks,
    top_n_regulators=4,
    minmax_scaling=True,
)

print("Signaling edges:")
print(active_signaling_network["sig"])
print("\nGene-regulatory edges:")
print(active_signaling_network["gr"])

### Format and visualize the signaling graph

We use `format_signaling_graph` to prepare node and edge tables, then visualize with networkx.

In [ ]:
graph_data = nn.format_signaling_graph(
    signaling_graph_list=active_signaling_network,
    ligands_all=ligands_oi,
    targets_all=targets_oi,
    sig_color="indianred",
    gr_color="steelblue",
)

print("Nodes:")
print(graph_data["nodes"])
print("\nEdges:")
print(graph_data["edges"])

In [ ]:
# Build a networkx DiGraph and visualize
G = nx.DiGraph()

node_colors = {}
for _, row in graph_data["nodes"].iterrows():
    G.add_node(row["label"])
    node_colors[row["label"]] = row["color"]

for _, row in graph_data["edges"].iterrows():
    G.add_edge(row["from"], row["to"], weight=row["weight"], color=row["color"])

fig, ax = plt.subplots(figsize=(10, 8))

pos = nx.spring_layout(G, seed=42)
colors = [node_colors.get(n, "grey") for n in G.nodes()]
edge_colors = [G[u][v]["color"] for u, v in G.edges()]
edge_widths = [G[u][v]["weight"] * 2 for u, v in G.edges()]

nx.draw(
    G, pos, ax=ax, with_labels=True,
    node_color=colors, edge_color=edge_colors,
    width=edge_widths, node_size=800,
    font_size=8, arrows=True, arrowsize=15,
)
ax.set_title("Ligand-to-target signaling path")
plt.tight_layout()
plt.show()

### Infer supporting data sources

We look at which of the collected data sources support the interactions in this network.

In [ ]:
data_source_network = nn.infer_supporting_datasources(
    signaling_graph_list=active_signaling_network,
    lr_network=lr_network,
    sig_network=sig_network,
    gr_network=gr_network,
)

data_source_network.head(10)

### Export for Cytoscape

You can export the networks for exploration in Cytoscape.

In [ ]:
write_output = False  # change to True for writing output
output_path = ""

if write_output:
    # Weighted signaling network
    sig_df = active_signaling_network["sig"].copy()
    sig_df["layer"] = "signaling"
    gr_df = active_signaling_network["gr"].copy()
    gr_df["layer"] = "regulatory"
    pd.concat([sig_df, gr_df]).to_csv(
        f"{output_path}weighted_signaling_network.txt", sep="\t", index=False
    )

    # Data source network
    data_source_network.to_csv(
        f"{output_path}data_source_network.txt", sep="\t", index=False
    )

    # Node annotation table
    all_genes = list(
        set(data_source_network["from"].tolist() + data_source_network["to"].tolist())
    )
    annotations = []
    for gene in all_genes:
        if gene in ligands_oi:
            annotations.append({"gene": gene, "annotation": "ligand"})
        elif gene in targets_oi:
            annotations.append({"gene": gene, "annotation": "target"})
        elif gene in lr_network["to"].unique():
            annotations.append({"gene": gene, "annotation": "receptor"})
        elif gene in gr_network["from"].unique():
            annotations.append({"gene": gene, "annotation": "transcriptional regulator"})
        else:
            annotations.append({"gene": gene, "annotation": "signaling mediator"})

    pd.DataFrame(annotations).to_csv(
        f"{output_path}annotation_table.txt", sep="\t", index=False
    )